# Dataset exploration - CIRA-CIC-DoHBrw-2020

**Purpose (Day 13):** know your data before modelling: class balance, feature ranges, which features separate benign from malicious. Put the dataset's Layer-2 CSVs in `ml/data/`. If that folder is empty this notebook falls back to the FAKE synthetic data and says so loudly.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# find the repo root (the folder that contains client/) no matter where Jupyter was started
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "client").is_dir())
for sub in ("client", "ml", "capture"):
    sys.path.insert(0, str(ROOT / sub))
RESULTS = ROOT / "results"
print("repo root:", ROOT)

In [ ]:
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from load_dataset import load_flows, find_csvs, prepare
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

DATA = ROOT / "ml" / "data"
if not any(DATA.rglob("*.csv")):
    DATA = ROOT / "ml" / "data_synthetic"
    print("!!! USING FAKE SYNTHETIC DATA - nothing here may be reported in the paper !!!")
X, y, info = prepare(load_flows(find_csvs(DATA)))
print("rows kept:", len(y), "| dropped (NaN/inf):", info["rows_dropped_nan_inf"], "| features:", X.shape[1])

## 1. Class balance
If one class dominates, accuracy is misleading - always report precision, recall, F1 and AUC too.

In [ ]:
counts = pd.Series(y).map({0: "Benign", 1: "Malicious"}).value_counts()
print(counts.to_string(), f"\nimbalance ratio: {counts.max() / counts.min():.2f} : 1")
counts.plot.bar(color=["tab:blue", "tab:red"], rot=0, title="Class balance"); plt.ylabel("flows"); plt.show()

## 2. Feature overview

In [ ]:
X.describe().T.round(4)

## 3. Which features separate the classes?
A quick, model-free look: median per class and their ratio (large ratio = informative).

In [ ]:
med = X.groupby(pd.Series(y).map({0: "benign", 1: "malicious"}).values).median().T
med["ratio_mal_over_benign"] = med["malicious"] / med["benign"].replace(0, np.nan)
med.sort_values("ratio_mal_over_benign", key=lambda s: (s - 1).abs(), ascending=False).head(12).round(4)

In [ ]:
top = med.sort_values("ratio_mal_over_benign", key=lambda s: (s - 1).abs(), ascending=False).index[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, col in zip(axes, top):
    sns.boxplot(x=pd.Series(y).map({0: "benign", 1: "malicious"}), y=np.log10(X[col].clip(lower=1e-9)), ax=ax, showfliers=False)
    ax.set_title(col, fontsize=9); ax.set_xlabel(""); ax.set_ylabel("log10(value)")
fig.tight_layout(); plt.show()

## 4. Redundant features
Very highly correlated pairs (|r| > 0.95) carry the same information.

In [ ]:
corr = X.corr().abs()
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1)).stack().sort_values(ascending=False))
pairs[pairs > 0.95].head(15).round(3)